---
title: Solusi latihan Panduan Pemula
short_title: Solusi latihan
subject: Panduan Pemula
subtitle: Solusi lengkap untuk latihan Panduan Pemula.
description: Solusi lengkap untuk latihan Panduan Pemula.
keywords:
  - open-data-cube
  - odc
  - xarray
  - plotting
  - spectral-indices
  - beginner-guide
  - exercise
---

Notebook ini berisi pembahasan lengkap latihan Panduan Pemula dengan wilayah kajian di sekitar Kuta, Lombok.[^edits]
Setiap langkah disertai alasan pengerjaan dan petunjuk untuk membaca hasilnya.

[^edits]: Notebook tutorial diperbarui secara otomatis; perubahan pada notebook tutorial dapat tertimpa pada pembaruan berikutnya.
    Salinan kerja perlu disimpan dalam file terpisah agar perubahan tetap tersimpan.

## A. Tujuan

- Mengubah titik menjadi wilayah kajian
- Menemukan dan memuat data satelit tahunan
- Memeriksa `xarray.Dataset`
- Membuat plot citra warna alami
- Menghitung dan membuat plot NDVI serta NDWI
- Membandingkan plot indeks tahunan

## B. Persiapan

`Datacube` digunakan untuk membuka koneksi data, sedangkan `point` membentuk geometri spasial dari pasangan bujur dan lintang.
Nama aplikasi mencatat identitas koneksi ini pada log datacube, lalu koneksinya disimpan sebagai `dc` agar dapat digunakan oleh kueri berikutnya.

In [ ]:
from datacube import Datacube
from odc.geo.geom import point

dc = Datacube(app="beginners_guide_exercise")

Sel ini tidak menampilkan keluaran karena koneksi hanya disimpan untuk dipakai pada langkah selanjutnya.
Jika koneksi gagal, masalah tersebut perlu diselesaikan sebelum pemuatan data dilanjutkan.

## C. Pemuatan data

Sebuah titik belum memiliki luas, sehingga perlu diperbesar lebih dahulu agar dapat menjadi wilayah permintaan data satelit.
Buffer `0.05` derajat membentuk wilayah tersebut, lalu `boundingbox` menghasilkan batas kiri, kanan, bawah, dan atas yang diperlukan oleh kueri.

In [ ]:
latitude = -8.81
longitude = 116.008667

bbox = point(longitude, latitude, crs="EPSG:4326").buffer(0.05).boundingbox
bbox.explore()

Peta yang muncul digunakan untuk memeriksa lokasi sebelum data dimuat.
Kotaknya semestinya mencakup kawasan sekitar Kuta; lokasi yang melenceng biasanya menandakan bahwa urutan bujur dan lintang tertukar.

Kueri meminta produk Sentinel-2 GeoMAD tahunan dari 2022 sampai 2025 beserta empat measurement yang diperlukan untuk citra warna alami dan indeks spektral.
Kuta berada di zona UTM 50S, sehingga `EPSG:32750` menghasilkan grid dengan satuan meter dan sesuai dengan resolusi 30 meter yang diminta.

In [ ]:
query = {
    "product": "s2_geomad_annual",
    "x": (bbox.left, bbox.right),
    "y": (bbox.bottom, bbox.top),
    "time": ("2022", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32750",
    "resolution": (-30, 30),
}

ds = dc.load(**query)

Data hasil pemuatan tersimpan dalam `ds` sebagai sebuah `xarray.Dataset`.
Penugasan tersebut tidak langsung menampilkan keluaran; isi Dataset diperiksa melalui sel-sel berikutnya.

## D. Pemeriksaan Dataset

Tampilan Dataset memberikan gambaran awal tentang dimensi, koordinat, measurement, dan atributnya.
Ekspresi berikutnya memisahkan informasi yang diminta dalam latihan agar lebih mudah diperiksa.

In [ ]:
ds.sizes

Pemetaan ukuran menunjukkan panjang setiap dimensi.
Dimensi `time` seharusnya berisi empat irisan tahunan, sedangkan `x` dan `y` menunjukkan jumlah piksel 30 meter yang meliputi wilayah kajian.

In [ ]:
ds.data_vars

Daftar variabel data seharusnya memuat `red`, `green`, `blue`, dan `nir`.
Semua variabel memakai dimensi `time`, `y`, dan `x` yang sama, sehingga nilainya dapat dipadukan piksel demi piksel.

In [ ]:
ds.attrs["crs"]

Keluaran CRS seharusnya menunjukkan `EPSG:32750`, sama dengan grid keluaran yang ditetapkan dalam kueri.

## E. Plot citra warna alami

Citra warna alami memerlukan kanal `red`, `green`, dan `blue` dalam satu larik.
`to_array` menumpuk ketiganya pada dimensi baru bernama `band`, lalu `isel(time=0)` mengambil komposit tahunan pertama.
Rentang tampilan 0 sampai 3000 mencegah segelintir piksel yang sangat terang menentukan kontras seluruh citra.

In [ ]:
rgb = ds[["red", "green", "blue"]].to_array(dim="band").isel(time=0)
rgb.plot.imshow(vmin=0, vmax=3000)

Warna pada citra ini menjadi acuan yang mudah dikenali ketika membaca plot indeks.
Daratan bervegetasi umumnya tampak hijau, perairan terbuka tampak gelap, sedangkan kawasan terbangun atau tanah terbuka terlihat lebih terang.
Karena data yang digunakan berupa komposit tahunan, tampilannya tidak selalu sama dengan citra pada satu tanggal tertentu.

## F. Perhitungan indeks spektral

NDVI membandingkan pantulan inframerah-dekat dan merah untuk menonjolkan vegetasi.
NDWI membandingkan pantulan hijau dan inframerah-dekat untuk menonjolkan air.
Band sumber diubah ke tipe floating-point sebelum pengurangan dan pembagian.
Langkah ini mencegah hasil yang keliru akibat operasi pada bilangan bulat, termasuk tipe `unsigned`, serta memastikan indeks dihitung sebagai nilai pecahan.

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}}$$

In [ ]:
red = ds.red.astype("float32")
green = ds.green.astype("float32")
nir = ds.nir.astype("float32")

ds["ndvi"] = (nir - red) / (nir + red)
ds["ndwi"] = (green - nir) / (green + nir)

ds[["ndvi", "ndwi"]]

Keluaran sel memastikan bahwa `ndvi` dan `ndwi` sudah ditambahkan ke Dataset untuk setiap irisan waktu dan piksel.
Sebagian besar nilai yang sah berada pada rentang -1 sampai 1, sedangkan lokasi tanpa data yang memadai tetap bernilai kosong.

## G. Plot indeks

Skala tetap dari -1 sampai 1 menjaga hubungan antara warna dan nilai agar sama pada seluruh plot.
Tanpa skala bersama, warna yang serupa pada panel berbeda belum tentu mewakili nilai indeks yang sama.

In [ ]:
ds.ndvi.isel(time=0).plot(cmap="RdYlGn", vmin=-1, vmax=1)

Pada plot NDVI, piksel yang semakin hijau memiliki nilai lebih tinggi dan umumnya menunjukkan vegetasi yang lebih rapat atau sehat.
Piksel kuning hingga merah memiliki nilai lebih rendah dan lebih mungkin berupa vegetasi jarang, tanah terbuka, kawasan terbangun, atau air.

In [ ]:
ds.ndwi.isel(time=0).plot(cmap="RdBu", vmin=-1, vmax=1)

Pada plot NDWI, piksel biru memiliki nilai lebih tinggi dan biasanya berkaitan dengan perairan terbuka.
Piksel merah memiliki nilai lebih rendah dan umumnya merupakan daratan.

In [ ]:
ds.ndvi.plot(col="time", col_wrap=2, cmap="RdYlGn", vmin=-1, vmax=1)

Facet grid menempatkan plot NDVI tiap tahun secara berdampingan dan memberi label waktu pada setiap panel.
Karena seluruh panel memakai cakupan wilayah dan skala yang sama, perubahan warna pada lokasi yang sama menunjukkan perbedaan komposit tahunan, bukan perubahan batas plot.

## H. Interpretasi hasil

1. Nilai NDVI tertinggi terdapat pada bagian daratan yang berwarna paling hijau dan ditutupi vegetasi rapat.
   Nilai terendah terdapat di laut serta pada tanah terbuka atau kawasan terbangun di sekitar Kuta.
2. Perairan terbuka pada umumnya memiliki NDWI tinggi sekaligus NDVI rendah.
   Pola yang berlawanan ini terjadi karena air menyerap inframerah-dekat dengan kuat, sehingga NDVI turun sementara NDWI naik.
3. Perubahan tahunan paling jelas terlihat pada piksel daratan yang menunjukkan perubahan warna besar antara merah, kuning, dan hijau pada panel NDVI.
   Petak lahan budidaya, lahan yang dibuka, atau kawasan yang sedang berkembang cenderung lebih banyak berubah daripada perairan terbuka atau vegetasi rapat yang menetap.
4. Sebagian perbedaan dapat menunjukkan perubahan tutupan lahan atau kondisi vegetasi.
   Sebagian lainnya dapat dipengaruhi oleh musim dan kumpulan pengamatan valid yang digunakan untuk menyusun setiap komposit GeoMAD tahunan.
   Karena itu, perubahan warna saja belum cukup untuk membuktikan adanya perubahan permanen di lapangan.

Citra warna alami perlu digunakan sebagai pembanding sebelum pola indeks ditafsirkan sebagai air, vegetasi, tanah terbuka, atau kawasan terbangun.

## I. Tantangan opsional

Susunan facet yang sama dapat diterapkan pada NDWI.
Perbandingan dengan facet grid NDVI menunjukkan apakah lokasi yang mengalami perubahan vegetasi juga mengalami perubahan NDWI.

In [ ]:
ds.ndwi.plot(col="time", col_wrap=2, cmap="RdBu", vmin=-1, vmax=1)

Pergeseran menuju biru berarti NDWI lebih tinggi, sedangkan pergeseran menuju merah berarti NDWI lebih rendah.
Skala yang tetap membuat perubahan tersebut dapat dibandingkan antartahun.

## J. Langkah selanjutnya

Mulai ulang kernel, lalu jalankan semua sel untuk memastikan bahwa solusi tidak bergantung pada kondisi yang tersimpan dari eksekusi sebelumnya.
Setelah seluruh sel berjalan tanpa kesalahan, bandingkan setiap bagian dengan latihan dan perhatikan pendekatan lain yang dapat menghasilkan keluaran yang sama.